# EXP-2026-008 C4 follow-up 3 on LUMIERE: enhancement at the border of the pre-op core (CPU)

Benign thin enhancement along the cavity wall is the most common early post-op finding in the literature, and it grows with the hours since surgery. This notebook runs the check fixed in card `experiments/registry/EXP-2026-008.md` (C4 follow-up 3) on the same 25 accepted training-role pairs of EXP-2026-005 run r2. Every step calls the tested module `rhuh_synthesis/lumiere_scaffold_check.py` (`run_wall_check`, which uses `rhuh_synthesis/core_signal.py`).

- **What it measures:**
  - Each pair's post-op T1, T1ce, T2 and FLAIR (DeepBraTumIA's atlas copies) are moved into the pre-op frame with the rigid matrix stored for its T1ce, as in follow-up 2.
  - A shell from 2 mm inside to 3 mm outside the pre-op core stands in for the cavity wall. Each shell voxel is called by the rule fixed in the card: "enhancing" means bright on T1ce and not on T1; "T1-bright" means bright on T1.
  - The same calls over far normal tissue give the background (vessels also enhance). Per pair, the excess is the shell share minus the background share.
- **Checks:** the same as follow-up 2. The cohort and the manifest must match run r2's records, each post-op T1ce file must match its SHA-256, and the moved T1ce must reproduce the export exactly. Anything else stops the run.
- **Inputs:**
  - The LUMIERE files are read through the Drive mount (`MyDrive/LUMIERE`), read only.
  - Run r2's records and exports, and run `c4_lumiere_regions_v1`'s record, are read through the Drive API by ID.
  - No folder is written except this run's own.
- **Output:** one write-once record and private review pages, uploaded through the Drive API to `MyDrive/LUMIERE_derived/EXP-2026-008/<RUN_TAG>` and confirmed by MD5.
  - The pages are `pages/wall_rank_NN.png`: the 3 complete resections with the largest excess and the 1 with the smallest. They show post-op T1ce and T1 with the shell in yellow and its enhancing voxels in red, plus the pre-op T1ce.
  - The pages and the record are private: they show and name patients.
  - The printed summary and table name no one.

A CPU runtime is enough. Expect roughly 10 to 20 minutes, as for follow-up 2. Colab asks for permission twice: the Drive mount and the Drive API. Keep the tab open until the last cell finishes.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-c4-wall"
LUMIERE_ROOT = "/content/drive/MyDrive/LUMIERE"  # read only, through the mount
RUN_TAG = "c4_lumiere_wall_v1"  # a new tag for every attempt; a run folder that holds files is refused
SOURCE_RUN = ("LUMIERE_derived", "EXP-2026-005", "r2")  # read only
REGIONS_RUN = ("LUMIERE_derived", "EXP-2026-008", "c4_lumiere_regions_v1")  # read only
OUT_RUN = ("LUMIERE_derived", "EXP-2026-008", RUN_TAG)  # created under MyDrive through the Drive API
EXPECTED_PAIRS = 25
LOCAL_SOURCE = "/content/c4_source_r2"
LOCAL_REGIONS = "/content/c4_lumiere_regions_v1_record"
LOCAL_RUN = f"/content/{RUN_TAG}"

## 1. Drive mount, code from the fork, packages, Drive API

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
assert Path(LUMIERE_ROOT).is_dir(), f"{LUMIERE_ROOT} not found: check the folder name"
assert not Path(LOCAL_RUN).exists(), f"{LOCAL_RUN} exists: choose a new RUN_TAG or a fresh runtime"
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

from rhuh_synthesis.colab_packages import DATA  # the one package list; it imports nothing

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DATA], check=True)
commit = subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip()
print(commit)

from google.colab import auth
from googleapiclient.discovery import build

from rhuh_synthesis.drive_api import DriveRun

auth.authenticate_user()  # Colab asks once for permission to use the Drive API with this account
api = build("drive", "v3", cache_discovery=False)
source = DriveRun.open(api, (), SOURCE_RUN, LOCAL_SOURCE, create=False)  # run r2, by ID; never written
regions_run = DriveRun.open(api, (), REGIONS_RUN, LOCAL_REGIONS, create=False)  # never written
run_drive = DriveRun.open(api, (), OUT_RUN, LOCAL_RUN, create=True)
access = run_drive.preflight()  # a real write, update, read and removal on Drive before the long part
print("free GiB on Drive:", None if access["free_gib"] is None else round(access["free_gib"], 1))
STARTED = time.perf_counter()


def say(message):
    print(f"[{time.perf_counter() - STARTED:6.0f} s] {message}", flush=True)

## 2. Run r2's records and exports, and the region record, by ID

In [ ]:
for name in ("run.json", "split.json", "registration_qc.csv", "summary.json", "manifest.json", "mask_check.json"):
    source.fetch(f"outputs/{name}")  # by ID, checked by MD5
outputs = Path(LOCAL_SOURCE) / "outputs"
r2 = json.loads((outputs / "summary.json").read_text(encoding="utf-8"))
assert r2["stopped_at"] is None and r2["exports"]["train"]["accepted"] == EXPECTED_PAIRS, "Run r2's summary differs from the card: stop and report"
say(f"run r2 training role: {r2['exports']['train']}")
files = source.fetch_folder("outputs/export/train")  # the training-role exports and their sidecars only
say(f"downloaded {files} files of the training-role exports")
regions_record = json.loads(regions_run.fetch("c4_lumiere_regions_v1.json").read_text(encoding="utf-8"))  # by ID
say(f"region record: {regions_record['summary']['rows']} pairs, source {regions_record['summary']['source']}")

## 3. The shell check on every accepted training-role pair, the record and the review pages

In [ ]:
from rhuh_synthesis.lumiere_scaffold_check import run_wall_check

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))  # frozen, read only
summary, table, path, digest = run_wall_check(
    outputs, LUMIERE_ROOT, config, regions_record, Path(LOCAL_RUN) / "outputs", source_run="EXP-2026-005 run r2",
    expected=EXPECTED_PAIRS, repo_root=WORK, progress=say,
)
report = run_drive.sync()
say(f"record {digest[:12]} written; Drive upload: {report}")

## 4. Confirm the upload and print the summary and the table

In [ ]:
assert run_drive.confirm(), f"Not every file is confirmed on Drive. The originals are in {LOCAL_RUN}: keep this session open and report"
print("Every file confirmed by the Drive server: MyDrive/" + "/".join(OUT_RUN))
print("SUMMARY")
print(json.dumps(summary, indent=1))
print("TABLE")
print(table)

## What to send back

Copy everything printed after `SUMMARY`, the `TABLE` included. They hold scores only, no patient name. If a cell stops with an error, send the error message; the checks stop the run on purpose when a file or record does not match.

The review pages are in `outputs/pages/` of the run folder on Drive. Each title gives the rank (as in the follow-up 2 table), the expert rating and the enhancing excess. Look at them yourself: is the red mostly a thin line along the cavity wall, or vessels, or the edge of the brain? They show patient images, so do not share them.